# Módulo 3 · Clase 4: Clasificación Multiclase — el Mapa de Facies del Campo

**Machine Learning for Petroleum Engineers Using Python**
SLB Ecuador · UDLA · 2026

Instructor: **Carlos Enrique Mosquera Trujillo**
Repo: [github.com/cmosquerat/slb-diplomado](https://github.com/cmosquerat/slb-diplomado)

---
## La idea de hoy

El subsuelo casi nunca pregunta "¿arena sí o no?". Pregunta **"¿cuál de todas?"**. Hoy quitamos la simplificación binaria de la Clase 2: el modelo debe elegir entre **9 facies**, medio pie a medio pie, usando solo registros de pozo.

En el camino aprenderemos tres cosas nuevas que valen para *cualquier* proyecto de ML:

1. **Codificar** variables categóricas (binarizar / `LabelEncoder` / one-hot) — el idioma que los modelos entienden.
2. **Leer 9 clases a la vez**: matriz de confusión 9×9, precision/recall por clase, F1 **macro vs weighted**.
3. El **costo de negocio en 9 dimensiones**: la matriz de penalización. Y de postre: `GridSearchCV`.

## El problema, acotado (regla de la casa)

| | |
|---|---|
| **Pregunta de negocio** | ¿Qué facies hay en cada medio pie de los pozos **sin núcleo**, usando solo sus registros? |
| **Target** | `Facies` — 9 categorías geológicas |
| **Features** | 5 registros (`GR`, `ILD_log10`, `DeltaPHI`, `PHIND`, `PE`) + contexto (`NM_M`, `RELPOS`, `Formation`) |
| **Métrica de éxito** | accuracy y F1 por clase — y al final, la **penalización total** |
| **Récord a batir** | el modelo tonto (siempre la facies más común): **23 % de aciertos** |

> **Dataset**: campo **Hugoton** (Kansas, EE. UU.), uno de los campos de gas más grandes de Norteamérica. Con estos mismos datos la SEG lanzó en 2016 la **primera competencia de machine learning en geociencias**.

---
# 0 · Preparación

Las mismas librerías de siempre, más **dos herramientas nuevas** que hoy presentamos: `LabelEncoder` (codificar el target) y `GridSearchCV` (buscar hiperparámetros). Todo lo demás ya lo conocen de las Clases 1–3.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report)

URL = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/hugoton_facies.csv"
df = pd.read_csv(URL)
print(df.shape)

## Primero lo primero: ¿qué es una *facies*?

Una **facies** es un "tipo de roca": un paquete de características (tamaño de grano, textura, minerales, fósiles) que aparecen juntas porque la roca se formó en el **mismo ambiente de depósito** — una playa, una laguna, un arrecife.

**¿Por qué le importa al ingeniero?** Porque la facies **controla la porosidad y la permeabilidad**: un mapa de facies es un mapa de **calidad de roca**. Con él se correlaciona entre pozos, se decide dónde completar y se construye el modelo del yacimiento.

La facies la describe un geólogo mirando el **núcleo**… cuando hay núcleo (es caro). Nuestro modelo intentará describirla **desde los registros**, que sí están en todos los pozos.

## Las 9 facies del dataset

| # | Código | Qué es | Ambiente |
|---|---|---|---|
| 1 | `SS` | Arenisca | no marino |
| 2 | `CSiS` | Limolita gruesa | no marino |
| 3 | `FSiS` | Limolita fina | no marino |
| 4 | `SiSh` | Limolita y lutita marina | marino |
| 5 | `MS` | Mudstone (caliza de grano muy fino) | marino |
| 6 | `WS` | Wackestone (caliza con matriz de lodo) | marino |
| 7 | `D` | Dolomita | marino |
| 8 | `PS` | Packstone–grainstone (caliza granular) | marino |
| 9 | `BS` | Bafflestone (caliza de algas) | marino |

> Ojo geológico: varias son **vecinas de grano** (una limolita gruesa y una fina se parecen muchísimo). Recuerden esto cuando veamos los errores del modelo.

---
# 1 · Exploración a fondo

Regla del módulo: **nadie modela datos que no ha mirado**. Vamos variable por variable.

In [ ]:
df.head()

## Las columnas, una a una (lectura física)

| Columna | Tipo | Qué nos dice |
|---|---|---|
| `Facies` | **target** (número 1–9) | la facies descrita en el núcleo |
| `Formation` | **categórica** (14 valores) | en qué formación está la muestra (A1 SH, B5 LM, …) |
| `Well Name` | identificador | de qué pozo viene la muestra |
| `Depth` | numérica | profundidad (ft) |
| `GR` | numérica | radioactividad natural: las **arcillas radian más** |
| `ILD_log10` | numérica | resistividad profunda (en log10): fluidos y litología |
| `DeltaPHI` | numérica | diferencia entre dos porosidades: sensible a litología |
| `PHIND` | numérica | porosidad promedio neutrón–densidad |
| `PE` | numérica | efecto fotoeléctrico: **huella de la mineralogía** |
| `NM_M` | **categórica** (1/2) | contexto: intervalo no marino (1) o marino (2) |
| `RELPOS` | numérica | posición relativa dentro del ciclo estratigráfico (1 = tope) |

In [ ]:
print(df.dtypes)
print()
print('Formaciones distintas:', df['Formation'].nunique())
print('Valores de NM_M:', sorted(df['NM_M'].unique()))

> 🤔 **Pregunta clave**: hay **tres columnas** que no son números "de verdad": `Facies` (números que en realidad son etiquetas), `Formation` (texto) y `NM_M` (1/2 que en realidad es una categoría). Los modelos **solo comen números con significado numérico**. ¿Qué hacemos con estas tres? *Guarden la pregunta: es la Sección 2.*

In [ ]:
df['Well Name'].value_counts()

> 🔧 **Mini-ejercicio 1**: ¿cuántas muestras tiene el pozo `CHURCHMAN BIBLE`? ¿Cuál es el pozo con **menos** muestras y por qué crees que es tan chico? (Pista: `Recruit F9` no es un pozo real — es un "pozo reclutado" que los organizadores armaron solo con ejemplos de la facies BS.)

In [ ]:
# Escribe tu solucion aqui

In [ ]:
df[['GR', 'ILD_log10', 'DeltaPHI', 'PHIND', 'PE', 'RELPOS']].describe().round(2)

**Lectura física del `describe()`** (como en cada clase):

- `GR` máximo = **361 API**: lutitas calientes, muy radioactivas. El promedio (65) es roca normal.
- `DeltaPHI` tiene **negativos**: normal, es una *diferencia* de porosidades.
- `RELPOS` va de 0 a 1: es una posición relativa, ya viene "empaquetada".
- ⚠️ Miren el `count` de `PE`: **3 232**, cuando el resto tiene 4 149. **Faltan casi mil valores.**

In [ ]:
print(df.isna().sum())
print()
print('Pozos afectados:', list(df[df['PE'].isna()]['Well Name'].unique()))

## Datos faltantes: una decisión, no un accidente

A **3 pozos** no les corrieron la herramienta de PE. Opciones:

1. **Quitar las filas** incompletas.
2. **Quitar la columna** `PE`.
3. **Imputar**: rellenar el hueco con un valor razonable.

> 🤔 **Pregunta clave**: ¿cuál elegirías tú, y qué información necesitas para decidir? *Antes de responder, miremos la **forma de la distribución** de PE — ahí vive la respuesta de "con qué valor rellenar".*

In [ ]:
pe = df['PE'].dropna()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(pe, bins=45, color='#2563EB', alpha=0.75, edgecolor='white')
ax.axvline(pe.mean(), color='#C82B40', lw=2.4, label=f'media = {pe.mean():.2f}')
ax.axvline(pe.median(), color='#16A34A', lw=2.4, label=f'mediana = {pe.median():.2f}')
ax.set_xlabel('PE (b/e)'); ax.set_ylabel('muestras')
ax.set_title(f'PE: sesgada a la derecha (skew = {pe.skew():.2f})')
ax.legend()
plt.show()

print('GR, para comparar: skew =', round(df['GR'].skew(), 2), '(mucho mas sesgada)')

## El menú de la imputación: qué usar en cada caso

| Método | Qué hace | Cuándo elegirlo | Riesgo |
|---|---|---|---|
| Quitar filas | descarta muestras incompletas | faltan pocas filas y sobran datos | perder pozos o clases enteras |
| Quitar columna | descarta la variable | la variable aporta poco | botar señal valiosa (¡PE!) |
| Media | rellena con el promedio | distribución **simétrica**, faltante aleatorio | la arrastran los extremos |
| Mediana | rellena con el valor central | distribución **sesgada** (como PE) | aplana la variabilidad |
| Por grupo | mediana *dentro* de cada formación o pozo | el grupo explica la variable | grupos chicos → ruido |
| Con modelo | predice el faltante desde las otras variables (KNN, regresión) | falta mucho y la variable es clave | complejidad + fugas |

**Regla práctica sobre la distribución**: si es **simétrica**, media o mediana dan casi igual; si tiene **cola** (como PE, y peor aún GR con skew 2.0), la media miente → usen **mediana**.

**Nuestra decisión de hoy**: quitar filas. Es honesta, simple, y nos deja 3 232 muestras de 8 pozos. Al final del notebook compararemos contra imputar — con números reales.

> ⚠️ Detalle fino que dejamos sembrado: si imputan, la media/mediana se calcula **solo con el train** (calcularla con todo el dataset es *fuga de información*). El porqué completo, en la Clase 5.

In [ ]:
df = df.dropna().reset_index(drop=True)
print('Nuevo shape:', df.shape)
print('Pozos restantes:', df['Well Name'].nunique())

> 🔧 **Mini-ejercicio 2**: calcula cuántas filas perdimos con el `dropna()` y qué **porcentaje** del dataset original representan. (El original tenía 4 149 filas.)

In [ ]:
# Escribe tu solucion aqui

## El balance de clases

En la Clase 2 el desbalance era 2 a 1. Veamos el de hoy — primero le damos **nombre** a cada facies para poder leerla.

In [ ]:
FACIES_NOMBRES = {1: 'SS', 2: 'CSiS', 3: 'FSiS', 4: 'SiSh', 5: 'MS',
                  6: 'WS', 7: 'D', 8: 'PS', 9: 'BS'}
COLORES = {'SS': '#EAB308', 'CSiS': '#D97706', 'FSiS': '#92400E',
           'SiSh': '#78716C', 'MS': '#94A3B8', 'WS': '#60A5FA',
           'D': '#C82B40', 'PS': '#2563EB', 'BS': '#1E3A8A'}
ORDEN = ['SS', 'CSiS', 'FSiS', 'SiSh', 'MS', 'WS', 'D', 'PS', 'BS']

df['FaciesNombre'] = df['Facies'].map(FACIES_NOMBRES)

conteo = df['FaciesNombre'].value_counts().reindex(ORDEN)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(conteo.index, conteo.values, color=[COLORES[f] for f in conteo.index])
for i, v in enumerate(conteo.values):
    ax.text(i, v + 10, str(v), ha='center')
ax.set_title('Balance de clases (tras dropna)')
ax.set_ylabel('muestras')
plt.show()

> 🤔 **Pregunta clave**: la clase más común tiene ~7 veces más muestras que la más rara (la dolomita `D`). ¿Qué le puede pasar al modelo con la dolomita? **Escriban su hipótesis** — la verificaremos en la sección de métricas.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
datos = [df[df['FaciesNombre'] == f]['GR'] for f in ORDEN]
bp = ax.boxplot(datos, labels=ORDEN, patch_artist=True, showfliers=False)
for patch, f in zip(bp['boxes'], ORDEN):
    patch.set_facecolor(COLORES[f])
ax.set_title('GR por facies: las arcillosas radian mas')
ax.set_ylabel('GR (API)')
plt.show()

**Lectura física**: la física responde — las facies con más arcilla (`FSiS`, `SiSh`) radian más; los carbonatos limpios (`PS`, `BS`), menos. Pero miren los **solapes**: con GR solo, `MS`, `WS` y `D` son casi indistinguibles. Una variable no alcanza: el modelo tendrá que **combinar** las cinco curvas más el contexto.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for f in ORDEN:
    s = df[df['FaciesNombre'] == f]
    ax.scatter(s['GR'], s['PHIND'], s=7, alpha=0.5, color=COLORES[f], label=f)
ax.set_xlabel('GR (API)'); ax.set_ylabel('PHIND (%)')
ax.set_xlim(0, 200); ax.set_ylim(0, 42)
ax.set_title('9 facies solapadas: esto NO se separa con una recta')
ax.legend(ncol=3, fontsize=8, markerscale=2)
plt.show()

Es el mismo mensaje de las **lunas** de la Clase 3, con datos reales y 9 colores: no hay recta — ni nueve rectas — que separen esto. Por eso el modelo de hoy vuelve a ser el **Random Forest**.

## El perfil de un pozo

Cada medio pie tiene sus registros **y** la facies descrita en núcleo. Grafiquemos un pozo completo — la columna de colores de la derecha es lo que el modelo deberá **reconstruir**.

In [ ]:
def perfil_pozo(nombre):
    w = df[df['Well Name'] == nombre].sort_values('Depth')
    fig, axes = plt.subplots(1, 4, figsize=(9, 7), sharey=True,
                             gridspec_kw={'width_ratios': [1, 1, 1, 0.5]})
    for ax, col, cc in [(axes[0], 'GR', '#16A34A'), (axes[1], 'PHIND', '#2563EB'),
                        (axes[2], 'PE', '#C82B40')]:
        ax.plot(w[col], w['Depth'], color=cc, lw=1)
        ax.set_xlabel(col)
    for _, fila in w.iterrows():
        axes[3].axhspan(fila['Depth'] - 0.25, fila['Depth'] + 0.25,
                        color=COLORES[fila['FaciesNombre']])
    axes[3].set_xticks([]); axes[3].set_xlabel('Facies')
    axes[0].set_ylabel('Profundidad (ft)')
    axes[0].invert_yaxis()
    fig.suptitle(f'Pozo {nombre}', fontweight='bold')
    plt.show()

perfil_pozo('SHRIMPLIN')

> 🔧 **Mini-ejercicio 3**: grafica el perfil del pozo `SHANKLE` con la función `perfil_pozo`. ¿Domina el ambiente marino (azules) o el no marino (cafés y amarillos)? Compara con `SHRIMPLIN`.

In [ ]:
# Escribe tu solucion aqui

---
# 2 · Codificar: el idioma de los modelos

Todo lo que hay dentro de un modelo son **sumas y comparaciones**. "Dolomita" no se puede sumar. Hay que convertir las categorías en números — pero *bien*, porque los números traen **orden y distancia de regalo**, y las categorías geológicas no tienen ni lo uno ni lo otro.

## La trampa del orden falso

Si numeramos arenisca = 1, caliza = 2, lutita = 3… el modelo se cree que la caliza está "a mitad de camino" entre arenisca y lutita. **Geológicamente falso.**

Nuestro dataset ya viene con la trampa puesta: `Facies` llega como números 1–9. Por eso lo primero que hicimos fue **devolverles su nombre** (`FaciesNombre`).

## El mapa completo de codificación

| Técnica | Qué hace | Se usa en | Hoy |
|---|---|---|---|
| **Binarizar** | 2 categorías → 0/1 | features de 2 valores | `NM_M` → `Marino` |
| **LabelEncoder** | N nombres → códigos 0…N−1 | el **target** multiclase | `FaciesNombre` |
| **One-hot** | 1 columna → N columnas de 0/1 | features de 3+ valores | `Formation` (14 valores) |

## LabelEncoder: el diccionario del target

Para el target sí usamos números 0…8 — pero como **código**, no como cantidad. El clasificador trata cada código como casilla separada; nunca hace cuentas con ellos.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['FaciesNombre'])

print('Diccionario:', list(le.classes_))
print('Ida:   ', df['FaciesNombre'].values[:3], '->', y[:3])
print('Vuelta:', y[:3], '->', le.inverse_transform(y[:3]))

> 🔧 **Mini-ejercicio 4**: ¿qué código le tocó a la dolomita `'D'`? Averígualo con `le.transform(['D'])`. ¿Y qué facies es el código 8? Usa `le.inverse_transform([8])`.

In [ ]:
# Escribe tu solucion aqui

## One-hot y binarizar: las features categóricas

- `Formation` tiene **14 valores** → `pd.get_dummies` la convierte en **14 columnas** de 0/1 (cada fila tiene un solo 1: su casilla). Sin orden, sin trampa.
- `NM_M` tiene solo 2 valores → basta **binarizar**: `Marino` = 0/1, igual que la lutita de la Clase 2.

> 🤔 **Pregunta clave**: si one-hot es tan bueno… ¿por qué no se lo aplicamos al **target**? *Porque el target se volvería 9 columnas y `scikit-learn` creería que son 9 problemas independientes de sí/no — podría responder "sí" a dos facies a la vez, o a ninguna. Una muestra tiene exactamente una facies: el target debe ser una sola columna de códigos.*

In [ ]:
onehot = pd.get_dummies(df['Formation'], prefix='Fm')
print('Formation: de 1 columna a', onehot.shape[1], 'columnas')
print(list(onehot.columns[:4]), '...')

df['Marino'] = (df['NM_M'] == 2).astype(int)

FEATURES_NUM = ['GR', 'ILD_log10', 'DeltaPHI', 'PHIND', 'PE', 'RELPOS']
X = pd.concat([df[FEATURES_NUM + ['Marino']], onehot], axis=1)
print('X final:', X.shape)

---
# 3 · Los modelos

## El split — con una palabra nueva: `stratify`

Con 9 clases y algunas muy raras, un split al azar podría dejar la dolomita casi sin muestras en el test. `stratify=y` obliga a que train y test conserven **las mismas proporciones de cada clase**.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('Train:', X_tr.shape, '| Test:', X_te.shape)

## El modelo tonto (nuestro récord a batir)

Como siempre: el modelo que no sabe nada apuesta **siempre a la clase más común**.

In [ ]:
mas_comun = np.bincount(y_tr).argmax()
pred_tonto = np.full_like(y_te, mas_comun)
acc_tonto = accuracy_score(y_te, pred_tonto)
print(f'Modelo tonto (siempre {le.classes_[mas_comun]}): accuracy = {acc_tonto:.3f}')

In [ ]:
logistica = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
logistica.fit(X_tr, y_tr)
pred_lo = logistica.predict(X_te)
print(f'Logistica multiclase: accuracy = {accuracy_score(y_te, pred_lo):.3f}')

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)
pred_rf = rf.predict(X_te)
print(f'Random Forest: accuracy = {accuracy_score(y_te, pred_rf):.3f}')

## El marcador

| Modelo | Accuracy |
|---|---|
| Modelo tonto (siempre `CSiS`) | 0.229 |
| Logística multiclase (escalada) | 0.594 |
| **Random Forest (200 árboles)** | **0.764** |

- **76.4 %** de aciertos exactos entre **9 opciones** (el azar puro daría 11 %).
- La **no linealidad** vuelve a pagar: +17 puntos sobre la recta, igual que en la Clase 3.
- El código fue **idéntico** al binario: `scikit-learn` contó las clases de `y` solo.

¿76 % es *bueno*? Con la accuracy sola **no se puede saber**. Antes de abrir la caja, un experimento pendiente…

## La prueba de la trampa ordinal

¿Y si hubiéramos tratado la facies como **número** (regresión sobre 1–9)? Hagamos el experimento del error:

In [ ]:
reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_tr, y_tr)
pred_reg = reg.predict(X_te)
print('Predicciones:', pred_reg[:6].round(2))

pred_reg_red = np.clip(np.round(pred_reg), 0, 8).astype(int)
print(f'Accuracy si redondeamos: {accuracy_score(y_te, pred_reg_red):.3f}')
print(f'Accuracy clasificando bien: {accuracy_score(y_te, pred_rf):.3f}')

¿Facies **4.37**? No existe. El modelo promedia categorías como si fueran barriles, y al redondear se queda en **0.40** contra **0.76**. Moraleja: **número ≠ cantidad**. Cuando la variable es una categoría, el problema es de clasificación. Siempre.

## ¿Y si hubiéramos imputado en vez de botar filas?

Lo prometido en la Sección 1: comparemos las estrategias de imputación **con números reales**. Repetimos todo el pipeline sobre el dataset completo (4 149 filas), rellenando PE de tres maneras.

In [ ]:
df_full = pd.read_csv(URL)
df_full['FaciesNombre'] = df_full['Facies'].map(FACIES_NOMBRES)

def accuracy_con(d):
    d = d.copy()
    d['Marino'] = (d['NM_M'] == 2).astype(int)
    ohl = pd.get_dummies(d['Formation'], prefix='Fm')
    Xl = pd.concat([d[FEATURES_NUM + ['Marino']].reset_index(drop=True),
                    ohl.reset_index(drop=True)], axis=1)
    yl = LabelEncoder().fit_transform(d['FaciesNombre'])
    a, b, c, dd = train_test_split(Xl, yl, test_size=0.3, random_state=42, stratify=yl)
    m = RandomForestClassifier(n_estimators=200, random_state=42).fit(a, c)
    return accuracy_score(dd, m.predict(b))

print('quitar filas   :', round(accuracy_con(df_full.dropna()), 3))
print('mediana global :', round(accuracy_con(df_full.assign(PE=df_full['PE'].fillna(df_full['PE'].median()))), 3))
print('media global   :', round(accuracy_con(df_full.assign(PE=df_full['PE'].fillna(df_full['PE'].mean()))), 3))

Las tres estrategias **empatan** (0.755–0.767): el faltante era ordenado (3 pozos completos) y PE tiene sesgo moderado. La lección no es "cuál gana", sino que la imputación **se decide mirando la distribución y el mecanismo del faltante** — no a ciegas. Y que imputar habría recuperado 917 filas y 2 pozos para entrenar.

---
# 4 · Leer 9 clases a la vez: métricas multiclase

## La matriz de confusión, versión 9×9

Se lee igual que la 2×2 de la Clase 2: **fila = lo que era**, **columna = lo que dijo el modelo**. Ordenamos las facies **geológicamente** (no marinas primero) — miren lo que aparece.

In [ ]:
idx = [list(le.classes_).index(f) for f in ORDEN]
cm = confusion_matrix(y_te, pred_rf)[np.ix_(idx, idx)]

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap='Reds')
ax.set_xticks(range(9), ORDEN); ax.set_yticks(range(9), ORDEN)
for i in range(9):
    for j in range(9):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > cm.max()*0.55 else 'black')
ax.set_xlabel('Facies predicha'); ax.set_ylabel('Facies real')
ax.set_title('Matriz de confusion del Random Forest (test)')
plt.show()

**Los errores se pegan a la diagonal**: el modelo confunde cada facies con sus **vecinas geológicas**. Verifiquémoslo contando las confusiones más frecuentes:

In [ ]:
pares = []
for i in range(9):
    for j in range(9):
        if i != j and cm[i, j] > 0:
            pares.append((cm[i, j], ORDEN[i], ORDEN[j]))
for n, a, b in sorted(pares, reverse=True)[:6]:
    print(f'{a:5s} -> predicho {b:5s}: {n} veces')

> 🤔 **Pregunta clave**: `CSiS` ↔ `FSiS` (limolita gruesa vs fina) y `PS` ↔ `WS` (packstone vs wackestone) son los errores más comunes. Vuelvan a la tabla de las 9 facies: ¿estos pares tienen sentido geológico? ¿Qué confundiría un geólogo con solo el registro en la mano?
>
> Un modelo que se equivoca "con sentido" inspira más confianza que uno que acierta más pero delira cuando falla. De los 229 errores del bosque, el **67 %** cae en una facies geológicamente vecina.

## Precision y recall, ahora por clase

Las definiciones de la Clase 2 no cambian — solo que ahora hay **un par por facies**. Todo sale de una línea:

In [ ]:
print(classification_report(y_te, pred_rf, target_names=le.classes_, digits=2))

**Cómo leerlo** (ejemplo con `MS`, mudstone):

- **recall 0.55** = de todos los mudstone reales, solo encontró el 55 %.
- **precision 0.77** = cuando dice "mudstone", acierta el 77 % de las veces.
- **support** = cuántas muestras reales hay en el test. Las clases con soporte chico (`D`=29, `BS`=48, `SiSh`=55, `MS`=65) son las que el modelo aprende peor: **el desbalance de la exploración ya está cobrando** — tal como hipotetizaron en la Sección 1.

## El modelo perezoso: la ceguera a las clases raras

Entrenemos a propósito un bosque **recortado** (`max_depth=3`) y comparemos su recall por clase con el bosque completo.

In [ ]:
rf_flojo = RandomForestClassifier(n_estimators=200, max_depth=3, random_state=42)
rf_flojo.fit(X_tr, y_tr)
pred_flojo = rf_flojo.predict(X_te)
print(f'Accuracy del perezoso: {accuracy_score(y_te, pred_flojo):.3f}  (suena aceptable...)')

rec_flojo = recall_score(y_te, pred_flojo, average=None)
rec_full = recall_score(y_te, pred_rf, average=None)
rec_flojo_o = [rec_flojo[list(le.classes_).index(f)] for f in ORDEN]
rec_full_o = [rec_full[list(le.classes_).index(f)] for f in ORDEN]

xpos = np.arange(9)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(xpos - 0.2, rec_flojo_o, 0.4, color='#6B7280', label='RF perezoso (max_depth=3)')
ax.bar(xpos + 0.2, rec_full_o, 0.4, color='#C82B40', label='RF completo')
ax.set_xticks(xpos, ORDEN)
ax.set_ylabel('Recall'); ax.legend()
ax.set_title('El modelo perezoso ignora las clases raras')
plt.show()

Mudstone y dolomita: **recall = 0.00**. Para el modelo perezoso, **no existen** — apuesta siempre a las clases grandes, la accuracy lo premia (0.56 "suena aceptable") y las clases raras lo pagan.

## Dos maneras de promediar 9 notas: macro vs weighted

Tenemos 9 valores de F1, uno por facies. ¿Cómo resumirlos en un solo número?

- **F1 weighted**: cada clase pesa según su **tamaño**. La nota de `CSiS` (222 muestras) manda; la de `D` (29) casi no cuenta. **Protege a las clases grandes** — se parece a la accuracy.
- **F1 macro**: promedio simple, **cada clase vale igual**. La dolomita opina con la misma voz que la limolita. **Protege a las clases raras.**

Cuenta rápida con 2 clases: una grande (900 muestras, F1 = 0.90) y una rara (100 muestras, F1 = 0.10):
- weighted = 0.9 × 0.90 + 0.1 × 0.10 = **0.82** ("va bien")
- macro = (0.90 + 0.10) / 2 = **0.50** ("hay un problema")

**El mismo modelo.** La brecha entre los dos números es un **detector de clases abandonadas**.

In [ ]:
for nombre, pred in [('RF completo', pred_rf), ('RF perezoso', pred_flojo)]:
    mac = f1_score(y_te, pred, average='macro')
    wei = f1_score(y_te, pred, average='weighted')
    print(f'{nombre:12s}: F1 macro = {mac:.3f} | F1 weighted = {wei:.3f} | brecha = {wei-mac:.3f}')

En el perezoso la brecha delata el abandono (0.40 vs 0.51); en el completo van parejas (0.754 vs 0.763): ninguna facies quedó del todo abandonada.

> 🔧 **Mini-ejercicio 5**: verifica la "cuenta rápida" de arriba con código: crea dos arrays de numpy con los F1 (`[0.90, 0.10]`) y los soportes (`[900, 100]`), y calcula macro y weighted a mano. ¿Coinciden con 0.50 y 0.82?

In [ ]:
# Escribe tu solucion aqui

---
# 5 · No todos los errores cuestan igual: la matriz de penalización

En la Clase 2 el costo de negocio tenía 2 números (FP y FN). Con 9 clases necesita una **tabla completa**: cuánto duele confundir *cada* facies con *cada* otra.

Nuestra regla, simple y geológica (la competencia SEG usaba una parecida):

- acertar = **0**
- facies **vecina** = **1** (limolita gruesa por fina: molesto)
- facies **lejana** = **3** (caliza por arenisca: grave)

In [ ]:
VECINAS = {'SS': ['CSiS'], 'CSiS': ['SS', 'FSiS'], 'FSiS': ['CSiS'],
           'SiSh': ['MS'], 'MS': ['SiSh', 'WS'], 'WS': ['MS', 'D'],
           'D': ['WS', 'PS'], 'PS': ['WS', 'D', 'BS'], 'BS': ['PS', 'D']}

P = np.zeros((9, 9), dtype=int)
for i, a in enumerate(ORDEN):
    for j, b in enumerate(ORDEN):
        P[i, j] = 0 if a == b else (1 if b in VECINAS[a] else 3)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(P, cmap='Reds', vmin=0, vmax=3)
ax.set_xticks(range(9), ORDEN); ax.set_yticks(range(9), ORDEN)
for i in range(9):
    for j in range(9):
        ax.text(j, i, P[i, j], ha='center', va='center',
                color='white' if P[i, j] == 3 else 'black')
ax.set_xlabel('Facies predicha'); ax.set_ylabel('Facies real')
ax.set_title('Matriz de penalizacion: acertar 0, vecina 1, lejana 3')
plt.show()

La penalización total se calcula **casilla por casilla**: matriz de confusión × matriz de penalización, y se suma. Es la extensión natural del costo de la Clase 2.

In [ ]:
def penalizacion_total(y_true, y_pred):
    c = confusion_matrix(y_true, y_pred)[np.ix_(idx, idx)]
    return int((c * P).sum())

for nombre, pred in [('Modelo tonto', pred_tonto), ('Logistica', pred_lo),
                     ('Random Forest', pred_rf)]:
    print(f'{nombre:14s}: penalizacion = {penalizacion_total(y_te, pred):5d} puntos')

In [ ]:
aciertos_relajados = 0
clases = list(le.classes_)
for yt, yp in zip(y_te, pred_rf):
    if yt == yp or clases[yp] in VECINAS[clases[yt]]:
        aciertos_relajados += 1

print(f'Accuracy exacta:      {accuracy_score(y_te, pred_rf):.3f}')
print(f'Accuracy con vecinas: {aciertos_relajados / len(y_te):.3f}')

## Cómo reportarlo al que decide

El gerente no quiere oír "F1 macro 0.754". Compare:

- ❌ *"El clasificador alcanzó un accuracy de 0.764 con F1 macro de 0.754 y una penalización agregada de 379 unidades."*
- ✅ *"De cada 100 pies interpretados, el modelo acierta la facies exacta en **76** y en **92** queda a lo sumo en la facies vecina. Donde más se equivoca es entre limolitas — el mismo punto ciego que tendría un intérprete con solo registros. En mudstone y dolomita (pocas muestras) recomendamos revisión humana."*

Decir también **dónde no confiar** es lo que separa al ingeniero del que solo corre `.fit()`.

## ¿Qué miró el bosque para lograrlo?

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
importancias.tail(10).plot.barh(ax=ax, color='#C82B40')
ax.set_title('Importancias del Random Forest (top 10)')
plt.show()

Los **5 registros** dominan — y `PE`, el que casi botamos por sus faltantes, es de los más usados. Las columnas codificadas aportan menos por separado, pero en conjunto valen ~7 puntos de accuracy (compruébenlo: entrenen un RF solo con `FEATURES_NUM`). Recordatorio de la Clase 3: importancia ≠ causalidad.

---
# 6 · Las perillas del modelo: parámetros vs hiperparámetros

Dos tipos de números viven en un modelo:

| | **Parámetros** | **Hiperparámetros** |
|---|---|---|
| ¿Quién los pone? | los aprende **el modelo solo** durante `.fit()` | los eligen **ustedes**, antes de entrenar |
| Ejemplos | los cortes de cada árbol ("¿GR > 78?"), los coeficientes de la logística | `max_depth`, `n_estimators`, `test_size` |

Analogía de taladro: el **peso sobre la broca** lo ajusta el perforador (perilla); el **desgaste** que va tomando la broca lo pone la formación (aprendido).

Giremos las perillas a mano para ver cuánto importan:

In [ ]:
print('n_estimators (con max_depth libre):')
for n in [1, 10, 50, 200]:
    m = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_tr, y_tr)
    print(f'  {n:4d} arboles -> accuracy {accuracy_score(y_te, m.predict(X_te)):.3f}')

print('max_depth (con 200 arboles):')
for d in [2, 5, 10, None]:
    m = RandomForestClassifier(n_estimators=200, max_depth=d, random_state=42).fit(X_tr, y_tr)
    print(f'  profundidad {str(d):4s} -> accuracy {accuracy_score(y_te, m.predict(X_te)):.3f}')

> 🤔 **Pregunta incómoda**: acabamos de elegir perillas **mirando el test**… ¿no es eso hacer trampa? **Sí lo es.** El test debe tocarse una sola vez, al final. La herramienta que busca perillas *sin* tocar el test es la siguiente.

## GridSearchCV: la búsqueda automática de perillas

Le damos la **parrilla** de valores; él prueba **todas** las combinaciones, validando cada una **5 veces dentro del train** (`cv=5`). El test queda intacto.

In [ ]:
parrilla = {'max_depth': [5, 10, 20, None],
            'n_estimators': [50, 200]}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    parrilla, cv=5)
grid.fit(X_tr, y_tr)

print('Mejor combinacion:', grid.best_params_)
print('Accuracy de validacion:', round(grid.best_score_, 3))
print('Accuracy en test del mejor:', round(accuracy_score(y_te, grid.predict(X_te)), 3))

In [ ]:
resultados = pd.DataFrame(grid.cv_results_)[
    ['param_max_depth', 'param_n_estimators', 'mean_test_score']]
resultados.sort_values('mean_test_score', ascending=False).round(3)

**Lectura honesta de la búsqueda**:

- La **profundidad** manda: de 5 a 20 niveles se ganan 14 puntos. Los árboles (50 vs 200) mueven décimas.
- Los dos primeros lugares empatan (0.755): a partir de cierta profundidad, todo da más o menos igual.
- El ganador en test da **0.755** — prácticamente nuestro 0.764 de siempre. Moraleja: **no siempre hay tesoro escondido**; la búsqueda sirve para *confirmarlo sin hacer trampa*.
- Fueron 8 combinaciones × 5 validaciones = **40 entrenamientos**… hechos por él.

> 🤔 ¿Por qué 5 pliegues? ¿Por qué funciona validar dentro del train? Eso **es** la Clase 5 (validación cruzada). También quedó pendiente validar con **pozos completos por fuera** — la deuda anotada desde la Clase 2.

---
# 7 · Prácticas

## 🧩 Práctica 1: la logística bajo la lupa

El bosque ya rindió cuentas clase por clase. Ahora háganle la misma auditoría a la logística:

1. Entrena la **logística multiclase** (con `StandardScaler`, como arriba) sobre las mismas features codificadas.
2. Saca su `classification_report`. ¿Qué facies tienen **recall** por debajo de 0.40?
3. Compara su **F1 macro** contra su **F1 weighted**. ¿La brecha delata clases abandonadas?
4. **Pregunta de negocio**: si el cliente pregunta "¿qué tan bueno es el modelo?", ¿qué número le darías y por qué ese?

In [ ]:
# Escribe tu solucion aqui

## 🧩 Práctica 2: tu propia matriz de penalización

Nuestra matriz (0/1/3) es geológica. Háganla **de negocio**:

1. Supongan que las facies de **buena roca almacén** son `SS`, `PS` y `BS`. Construyan una matriz donde confundir roca almacén con roca sello (o al revés) cueste **5**, y cualquier otro error cueste **1**.
2. Calculen la **penalización total** del Random Forest y de la logística con *su* matriz. ¿Cambia el ganador?
3. Encuentren en la matriz de confusión **la casilla que más plata les cuesta** con su regla. ¿Entre qué facies está?
4. **Pregunta de negocio**: escriban **una frase para el gerente** de activos recomendando (o no) usar el modelo para mapear roca almacén.
5. **Bonus**: con `GridSearchCV`, agreguen `min_samples_leaf: [1, 5, 20]` a la parrilla. ¿Cambia el ganador?

In [ ]:
# Escribe tu solucion aqui

---
# Cierre

## Lo que aprendimos hoy

- **Multiclase**: elegir una entre N — mismo código, nueva lectura.
- **Faltantes e imputación**: el menú completo, y cómo la **distribución** (media vs mediana) decide el relleno.
- **Codificar**: binarizar / `LabelEncoder` / one-hot — y la **trampa del orden falso** (la facies 4.37 no existe).
- **Matriz 9×9** leída geológicamente: los errores del bosque caen en facies vecinas (67 %).
- **Macro vs weighted**: a quién protege cada promedio; la brecha como detector de clases abandonadas.
- **Matriz de penalización**: el costo de negocio en 9 dimensiones (tonto 1 718 → bosque 379).
- **Parámetros vs hiperparámetros**, y `GridSearchCV` para buscar perillas sin tocar el test.

## El número del día

> De cada 100 pies, el modelo acierta la facies exacta en **76** y en **92** queda a lo sumo en la vecina.

## Lo que sigue (Clase 5 — cierre del módulo)

- **SVM y el truco del kernel**: separar lo inseparable subiendo de dimensión (las lunas vuelven por última vez).
- **El porqué de la validación cruzada**: abrimos el `cv=5` de hoy.
- **La deuda pendiente**: validar con **pozos completos por fuera** (`GroupKFold`).